<br><h1>Fake News Detection Part 3</h1><br>

In [108]:
import polars as pl
import numpy as np

# Read in data
df = pl.read_csv("../news/data/995,000_rows_preprocessed.csv", columns=["type", "content"], n_rows=10_000)
df.head()

type,content
str,str
"""political""","""plus one articl googl plus tha…"
"""fake""","""cost best senat bank committe …"
"""satire""","""man awoken <NUM> -year coma co…"
"""reliable""","""julia geist ask draw pictur co…"
"""conspiracy""","""<NUM> compil studi vaccin dang…"


In [109]:
real_labels = ['reliable', 'political']
drop_labels = ['unknown', 'satire', 'bias', 'clickbait', 'state', 'hate', 'nan']
# This could effect real world use on data containing biased, satire etc. articles
# Also makes the training set smaller

df = df.filter(
    ~pl.col("type").is_in(drop_labels)).with_columns(
    pl.when(pl.col("type").is_in(real_labels))
    .then(pl.lit("real"))
    .otherwise(pl.lit("fake"))
    .alias("label")
)

In [110]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer as _TV

n = df.shape[0]
indices = np.arange(n)
labels = df["label"].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=1, stratify=labels
)
# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=1, stratify=labels[temp_idx]
)

df_train = df[train_idx.tolist()]
df_val = df[val_idx.tolist()]
df_test = df[test_idx.tolist()]

print(f"Train: {df_train.shape[0]}  ({df_train.shape[0] / n:.0%})")
print(f"Val:   {df_val.shape[0]}  ({df_val.shape[0] / n:.0%})")
print(f"Test:  {df_test.shape[0]}  ({df_test.shape[0] / n:.0%})")


Train: 5949  (80%)
Val:   744  (10%)
Test:  744  (10%)


In [111]:
# Verify that stratified splitting preserved the label ratio
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    dist = (
        split.group_by("label")
        .agg(pl.len().alias("count"))
        .sort("label")
        .with_columns(
            (pl.col("count").cast(pl.Float64) / pl.col("count").sum() * 100)
            .round(1)
            .alias("pct")
        )
    )
    print(f"\n{name} split:")
    print(dist)


Train split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 2812  ┆ 47.3 │
│ real  ┆ 3137  ┆ 52.7 │
└───────┴───────┴──────┘

Val split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 352   ┆ 47.3 │
│ real  ┆ 392   ┆ 52.7 │
└───────┴───────┴──────┘

Test split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 351   ┆ 47.2 │
│ real  ┆ 393   ┆ 52.8 │
└───────┴───────┴──────┘


In [112]:
from sklearn.feature_extraction.text import TfidfVectorizer
import gc

# Configure the TF-IDF vectorizer with uni- and bigrams
vectorizer = TfidfVectorizer(
    max_features=100_000,  # vocabulary cap
    ngram_range=(1, 2),  # unigrams
    sublinear_tf=True,  # apply 1 + log(tf)
    min_df=3,  # ignore terms in < 3 documents
    max_df=0.95,  # ignore terms in > 95 % of documents
    strip_accents="unicode",  # normalise accented characters
    token_pattern=r"[a-zA-Z]{2,}",  # only alphabetic tokens ≥ 2 chars
)
# Extract labels before we start deleting things
y_train = (df_train["label"] == "fake").cast(pl.Int8).to_numpy()
y_val   = (df_val["label"]   == "fake").cast(pl.Int8).to_numpy()
y_test  = (df_test["label"]  == "fake").cast(pl.Int8).to_numpy()

# Vectorize train, then drop the df
tfidf_train = vectorizer.fit_transform(df_train["content"])
del df_train; gc.collect()

tfidf_val = vectorizer.transform(df_val["content"])
del df_val; gc.collect()

tfidf_test = vectorizer.transform(df_test["content"])
del df_test; gc.collect()

# Also drop the full df if still around (Let the memory be freed!!)
del df; gc.collect()

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"TF-IDF train: {tfidf_train.shape}")
print(f"TF-IDF val:   {tfidf_val.shape}")
print(f"TF-IDF test:  {tfidf_test.shape}")

Vocabulary size: 96,399
TF-IDF train: (5949, 96399)
TF-IDF val:   (744, 96399)
TF-IDF test:  (744, 96399)


In [113]:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=1024)
# X_train = pca.fit_transform(tfidf_train)
# X_val = pca.transform(tfidf_val)
# X_test = pca.transform(tfidf_test)

In [114]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Try a logarithmic sweep of C values and pick the best one on the validation set
c_values = np.logspace(-4, 1, 8)
tuning_rows = []
best_c = None
best_f1 = -1.0

for c_value in c_values:
    candidate = LinearSVC(C=float(c_value), max_iter=10_000, random_state=1, dual="auto")
    candidate.fit(tfidf_train, y_train)

    y_val_candidate = candidate.predict(tfidf_val)
    val_accuracy = accuracy_score(y_val, y_val_candidate)
    val_f1 = f1_score(y_val, y_val_candidate)

    tuning_rows.append({
        "C": float(c_value),
        "val_accuracy": val_accuracy,
        "val_f1": val_f1,
    })

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_c = float(c_value)

tuning_results = (
    pl.DataFrame(tuning_rows)
    .sort(["val_f1", "val_accuracy"], descending=[True, True])
)
print(tuning_results)

# Refit one final model using the selected C
svm = LinearSVC(C=best_c, max_iter=10_000, random_state=1, dual="auto")
svm.fit(tfidf_train, y_train)

y_val_pred = svm.predict(tfidf_val)
print(f"\nBest C: {best_c}")
print("Validation:")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"F1: {f1_score(y_val, y_val_pred):.4f}")
print(classification_report(y_val, y_val_pred, target_names=["real", "fake"]))

shape: (8, 3)
┌──────────┬──────────────┬──────────┐
│ C        ┆ val_accuracy ┆ val_f1   │
│ ---      ┆ ---          ┆ ---      │
│ f64      ┆ f64          ┆ f64      │
╞══════════╪══════════════╪══════════╡
│ 0.372759 ┆ 0.915323     ┆ 0.909871 │
│ 0.071969 ┆ 0.907258     ┆ 0.901569 │
│ 1.930698 ┆ 0.901882     ┆ 0.895265 │
│ 10.0     ┆ 0.896505     ┆ 0.889209 │
│ 0.013895 ┆ 0.893817     ┆ 0.886331 │
│ 0.002683 ┆ 0.787634     ┆ 0.723776 │
│ 0.000518 ┆ 0.547043     ┆ 0.081744 │
│ 0.0001   ┆ 0.526882     ┆ 0.0      │
└──────────┴──────────────┴──────────┘

Best C: 0.3727593720314942
Validation:
Accuracy: 0.9153
F1: 0.9099
              precision    recall  f1-score   support

        real       0.91      0.93      0.92       392
        fake       0.92      0.90      0.91       352

    accuracy                           0.92       744
   macro avg       0.92      0.91      0.92       744
weighted avg       0.92      0.92      0.92       744

